# Chapter 10 - Logistic Regression

Linear regression predicts continuous values, but many real-world problems require
predicting **categories**: will the customer churn? Is this email spam? Does the patient
have the disease? Logistic regression is the workhorse algorithm for binary classification.
Despite its name, it is a **classification** method that models the probability of an
outcome belonging to a particular class.

**Topics covered:**
- Why linear regression fails for classification
- The logistic (sigmoid) function
- Log-odds and the decision boundary
- Maximum Likelihood Estimation (intuition)
- Binary classification with scikit-learn
- Interpreting coefficients as log-odds ratios
- Predicting probabilities with `predict_proba()`
- Tuning the decision threshold
- Visualizing decision boundaries on 2D data

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True

## From Regression to Classification: Why Linear Regression Fails

It is tempting to use linear regression for a binary outcome by treating the label as a
number (0 or 1) and fitting a line. Let us see why this breaks down.

In [ ]:
# Simulate a simple dataset: hours studied vs pass/fail
np.random.seed(42)
hours = np.concatenate([np.random.normal(3, 1.2, 30), np.random.normal(7, 1.2, 30)])
passed = np.array([0] * 30 + [1] * 30)

# Fit linear regression on binary labels
lin_reg = LinearRegression()
lin_reg.fit(hours.reshape(-1, 1), passed)

x_range = np.linspace(0, 12, 200).reshape(-1, 1)
y_pred_linear = lin_reg.predict(x_range)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(hours, passed, c=passed, cmap='bwr', edgecolors='black', s=60, zorder=3)
ax.plot(x_range, y_pred_linear, color='green', linewidth=2, label='Linear regression')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.axhline(1, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Hours Studied')
ax.set_ylabel('Predicted Value')
ax.set_title('Linear Regression on Binary Labels')
ax.legend()
plt.tight_layout()
plt.show()

**Problems with using linear regression for classification:**

1. **Predictions outside [0, 1]:** The line extends to negative values and values greater
   than 1, which cannot be interpreted as probabilities.
2. **Sensitive to outliers:** A single extreme data point far to the right can tilt the
   entire regression line, changing the decision boundary for all points.
3. **Assumes linear relationship with the label:** The true relationship between features
   and a binary outcome is inherently non-linear — it must be an S-shaped curve that
   saturates at 0 and 1.

We need a function that maps any real number to the range (0, 1). Enter the **sigmoid function**.

## The Logistic (Sigmoid) Function

The sigmoid function is defined as:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

where $z$ can be any real number. Key properties:

- **Output range:** Always between 0 and 1 — perfect for modeling probabilities
- **Symmetry:** $\sigma(-z) = 1 - \sigma(z)$
- **Center:** $\sigma(0) = 0.5$
- **Limits:** As $z \to +\infty$, $\sigma(z) \to 1$; as $z \to -\infty$, $\sigma(z) \to 0$

In [ ]:
def sigmoid(z):
    """Compute the sigmoid function."""
    return 1 / (1 + np.exp(-z))

z = np.linspace(-8, 8, 300)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(z, sigmoid(z), color='blue', linewidth=2.5, label=r'$\sigma(z) = \frac{1}{1 + e^{-z}}$')
ax.axhline(0.5, color='red', linestyle='--', alpha=0.6, label='y = 0.5 (decision boundary)')
ax.axvline(0, color='gray', linestyle='--', alpha=0.4)
ax.set_xlabel('z', fontsize=12)
ax.set_ylabel(r'$\sigma(z)$', fontsize=12)
ax.set_title('The Sigmoid Function', fontsize=14)
ax.legend(fontsize=11)
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

In logistic regression, the input $z$ to the sigmoid is the familiar linear combination:

$$z = w_0 + w_1 x_1 + w_2 x_2 + \cdots + w_n x_n = \mathbf{w}^T \mathbf{x}$$

So the predicted probability of the positive class is:

$$P(y=1 \mid \mathbf{x}) = \sigma(\mathbf{w}^T \mathbf{x}) = \frac{1}{1 + e^{-\mathbf{w}^T \mathbf{x}}}$$

The model learns the weights $\mathbf{w}$ that best separate the classes.

### How the Sigmoid Shape Changes with Weights

The weight controls the **steepness** of the sigmoid. A larger weight means the model is
more "confident" — the transition from 0 to 1 happens more sharply.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for w in [0.5, 1.0, 2.0, 5.0]:
    ax.plot(z, sigmoid(w * z), linewidth=2, label=f'w = {w}')

ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('x')
ax.set_ylabel(r'$\sigma(w \cdot x)$')
ax.set_title('Effect of Weight Magnitude on the Sigmoid')
ax.legend()
plt.tight_layout()
plt.show()

## Log-Odds and the Decision Boundary

The **odds** of an event with probability $p$ are:

$$\text{odds} = \frac{p}{1 - p}$$

The **log-odds** (also called the **logit**) are the natural logarithm of the odds:

$$\text{logit}(p) = \ln\left(\frac{p}{1 - p}\right)$$

The remarkable insight of logistic regression is that the logit is a **linear function** of the features:

$$\ln\left(\frac{P(y=1)}{1 - P(y=1)}\right) = \mathbf{w}^T \mathbf{x}$$

This means logistic regression is really a **linear model in log-odds space**. The sigmoid
just converts this linear output into a probability.

In [ ]:
# Visualize probability, odds, and log-odds side by side
p = np.linspace(0.01, 0.99, 200)
odds = p / (1 - p)
log_odds = np.log(odds)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(p, p, 'b-', linewidth=2)
axes[0].set_xlabel('p')
axes[0].set_ylabel('Probability')
axes[0].set_title('Probability (bounded 0-1)')

axes[1].plot(p, odds, 'r-', linewidth=2)
axes[1].set_xlabel('p')
axes[1].set_ylabel('Odds = p / (1-p)')
axes[1].set_title('Odds (bounded 0-inf)')
axes[1].set_ylim(0, 20)

axes[2].plot(p, log_odds, 'g-', linewidth=2)
axes[2].set_xlabel('p')
axes[2].set_ylabel('log(p / (1-p))')
axes[2].set_title('Log-Odds (unbounded)')

plt.tight_layout()
plt.show()

### The Decision Boundary

The **decision boundary** is where the model is equally uncertain — the predicted
probability is exactly 0.5. At this point:

$$P(y=1) = 0.5 \implies \text{logit}(0.5) = 0 \implies \mathbf{w}^T \mathbf{x} = 0$$

So the decision boundary is the hyperplane $\mathbf{w}^T \mathbf{x} = 0$. For 2D data with features
$x_1$ and $x_2$, this is a **straight line**:

$$w_0 + w_1 x_1 + w_2 x_2 = 0 \quad \Rightarrow \quad x_2 = -\frac{w_0 + w_1 x_1}{w_2}$$

## Maximum Likelihood Estimation (Intuition)

How does logistic regression learn the weights? Unlike linear regression which minimizes
squared error, logistic regression uses **Maximum Likelihood Estimation (MLE)**.

The idea is intuitive: **find the weights that make the observed data most probable.**

For a single training example $(\mathbf{x}_i, y_i)$:
- If $y_i = 1$, we want $P(y=1 \mid \mathbf{x}_i) = \hat{p}_i$ to be **high**
- If $y_i = 0$, we want $P(y=0 \mid \mathbf{x}_i) = 1 - \hat{p}_i$ to be **high**

This can be written compactly as:

$$L(\mathbf{w}) = \prod_{i=1}^{n} \hat{p}_i^{y_i} (1 - \hat{p}_i)^{1 - y_i}$$

Taking the log (for numerical stability and to turn products into sums) gives the
**log-likelihood**, and negating it gives the **log-loss** (binary cross-entropy) that
scikit-learn minimizes:

$$\mathcal{L}(\mathbf{w}) = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \ln(\hat{p}_i) + (1 - y_i) \ln(1 - \hat{p}_i) \right]$$

In [ ]:
# Visualize the log-loss for a single sample
p_hat = np.linspace(0.001, 0.999, 300)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# When true label y = 1: loss = -log(p_hat)
axes[0].plot(p_hat, -np.log(p_hat), 'b-', linewidth=2)
axes[0].set_xlabel(r'Predicted probability $\hat{p}$')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss when y = 1 (want high $\hat{p}$)')
axes[0].annotate('Low loss\n(correct)', xy=(0.9, -np.log(0.9)), fontsize=10,
                 arrowprops=dict(arrowstyle='->', color='green'),
                 xytext=(0.6, 1.5), color='green')

# When true label y = 0: loss = -log(1 - p_hat)
axes[1].plot(p_hat, -np.log(1 - p_hat), 'r-', linewidth=2)
axes[1].set_xlabel(r'Predicted probability $\hat{p}$')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss when y = 0 (want low $\hat{p}$)')
axes[1].annotate('Low loss\n(correct)', xy=(0.1, -np.log(0.9)), fontsize=10,
                 arrowprops=dict(arrowstyle='->', color='green'),
                 xytext=(0.35, 1.5), color='green')

for ax in axes:
    ax.set_ylim(0, 5)

plt.suptitle('Binary Cross-Entropy Loss', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

> **Key intuition:** The log-loss penalizes confident wrong predictions **very harshly**.
> If the model predicts $\hat{p} = 0.01$ for a sample that is actually positive ($y=1$),
> the loss is $-\ln(0.01) \approx 4.6$. Being wrong and confident is extremely costly.

## Binary Classification with scikit-learn

Let us build a real classifier. We will generate a synthetic dataset with `make_classification`
and train a `LogisticRegression` model.

In [ ]:
# Generate synthetic binary classification data
X, y = make_classification(
    n_samples=500,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    random_state=42,
)

# Visualize
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='bwr', edgecolors='black', s=40, alpha=0.7)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_title('Synthetic Binary Classification Dataset')
plt.colorbar(scatter, label='Class')
plt.tight_layout()
plt.show()

print(f'Dataset shape: {X.shape}')
print(f'Class distribution: {np.bincount(y)}')

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'Train class balance: {np.bincount(y_train)}')
print(f'Test class balance:  {np.bincount(y_test)}')

In [ ]:
# Fit logistic regression
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print()
print(classification_report(y_test, y_pred))

## Interpreting Coefficients as Log-Odds Ratios

Each coefficient $w_j$ tells us: **a one-unit increase in feature $x_j$ changes the
log-odds of the positive class by $w_j$**, holding other features constant.

Equivalently, the odds are multiplied by $e^{w_j}$. This is the **odds ratio**.

In [ ]:
print('Model coefficients (log-odds):')
print(f'  Intercept (w0): {model.intercept_[0]:.4f}')
for i, coef in enumerate(model.coef_[0]):
    print(f'  Feature {i+1} (w{i+1}): {coef:.4f}  ->  odds ratio: {np.exp(coef):.4f}')

print()
print('Interpretation:')
for i, coef in enumerate(model.coef_[0]):
    direction = 'increases' if coef > 0 else 'decreases'
    pct = (np.exp(abs(coef)) - 1) * 100
    print(f'  A 1-unit increase in Feature {i+1} {direction} the odds by {pct:.1f}%')

## Predicting Probabilities with `predict_proba()`

The `predict()` method returns hard class labels (0 or 1). But often we need the
underlying **probabilities** — for ranking, for adjusting thresholds, or for understanding
model confidence. That is what `predict_proba()` provides.

In [ ]:
# predict_proba returns [P(class=0), P(class=1)] for each sample
probas = model.predict_proba(X_test)

print('First 10 predictions:')
print(f'{"P(class=0)":>12}  {"P(class=1)":>12}  {"Predicted":>10}  {"Actual":>8}')
print('-' * 50)
for i in range(10):
    print(f'{probas[i, 0]:>12.4f}  {probas[i, 1]:>12.4f}  {y_pred[i]:>10}  {y_test.iloc[i] if hasattr(y_test, "iloc") else y_test[i]:>8}')

In [ ]:
# Distribution of predicted probabilities by true class
fig, ax = plt.subplots(figsize=(10, 5))

for cls, color, label in [(0, 'blue', 'Class 0'), (1, 'red', 'Class 1')]:
    mask = y_test == cls
    ax.hist(probas[mask, 1], bins=20, alpha=0.6, color=color, label=label, edgecolor='black')

ax.axvline(0.5, color='black', linestyle='--', linewidth=2, label='Default threshold (0.5)')
ax.set_xlabel('Predicted P(class=1)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Predicted Probabilities by True Class')
ax.legend()
plt.tight_layout()
plt.show()

> A well-calibrated model will show class 0 samples clustered near low probabilities and
> class 1 samples near high probabilities, with minimal overlap.

## The Decision Threshold

By default, scikit-learn uses a threshold of **0.5**: if $P(y=1) \geq 0.5$, predict class 1.
But this is not always the best choice.

- In **medical diagnosis**, we might lower the threshold (e.g., 0.3) to catch more true
  positives, even at the cost of more false alarms.
- In **spam detection**, we might raise the threshold (e.g., 0.7) to avoid marking
  legitimate emails as spam.

The threshold is a **business decision**, not a mathematical one.

In [ ]:
# Compare different thresholds
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

print(f'{"Threshold":>10}  {"Accuracy":>10}  {"Predicted 1s":>14}  {"True 1s":>10}')
print('-' * 50)

true_positives = y_test.sum() if hasattr(y_test, 'sum') else sum(y_test)

for t in thresholds:
    y_pred_t = (probas[:, 1] >= t).astype(int)
    acc = accuracy_score(y_test, y_pred_t)
    predicted_pos = y_pred_t.sum()
    print(f'{t:>10.1f}  {acc:>10.4f}  {predicted_pos:>14}  {true_positives:>10}')

> Lowering the threshold increases the number of positive predictions (higher recall)
> but also increases false positives (lower precision). We will explore this tradeoff
> in depth in the next notebook.

## Visualizing the Decision Boundary

Since our data has only 2 features, we can plot the decision boundary directly in 2D.
The boundary is where the model's predicted probability equals the threshold.

In [ ]:
def plot_decision_boundary(model, X, y, title='Decision Boundary', ax=None):
    """Plot the decision boundary for a 2D classifier."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 7))

    # Create a mesh grid
    margin = 0.5
    x_min, x_max = X[:, 0].min() - margin, X[:, 0].max() + margin
    y_min, y_max = X[:, 1].min() - margin, X[:, 1].max() + margin
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300),
    )

    # Predict on the grid
    Z = model.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1]
    Z = Z.reshape(xx.shape)

    # Plot probability surface
    contour = ax.contourf(xx, yy, Z, levels=50, cmap='RdBu_r', alpha=0.8)
    ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)

    # Plot data points
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c='blue', edgecolors='black', s=40, label='Class 0')
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c='red', edgecolors='black', s=40, label='Class 1')

    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.set_title(title)
    ax.legend()

    return contour

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
contour = plot_decision_boundary(model, X, y, title='Logistic Regression Decision Boundary', ax=ax)
plt.colorbar(contour, ax=ax, label='P(class=1)')
plt.tight_layout()
plt.show()

> The decision boundary is a **straight line** — logistic regression is a linear classifier.
> The color gradient shows the predicted probability, transitioning smoothly from 0
> (blue) to 1 (red). The black line marks the 0.5 threshold.

### Effect of Threshold on the Boundary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, threshold in zip(axes, [0.3, 0.5, 0.7]):
    # Create mesh
    margin = 0.5
    x_min, x_max = X[:, 0].min() - margin, X[:, 0].max() + margin
    y_min, y_max = X[:, 1].min() - margin, X[:, 1].max() + margin
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300),
    )
    Z = model.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

    ax.contourf(xx, yy, Z, levels=50, cmap='RdBu_r', alpha=0.8)
    ax.contour(xx, yy, Z, levels=[threshold], colors='black', linewidths=2)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c='blue', edgecolors='black', s=20)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c='red', edgecolors='black', s=20)
    ax.set_title(f'Threshold = {threshold}')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

### Connecting Back to the Exam Example

Let us revisit our hours-studied example, this time with logistic regression.

In [ ]:
# Fit logistic regression on the hours-studied data
log_reg = LogisticRegression(random_state=42)
log_reg.fit(hours.reshape(-1, 1), passed)

x_range = np.linspace(0, 12, 300).reshape(-1, 1)
y_prob = log_reg.predict_proba(x_range)[:, 1]
y_linear = lin_reg.predict(x_range)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(hours, passed, c=passed, cmap='bwr', edgecolors='black', s=60, zorder=3)
ax.plot(x_range, y_linear, color='green', linewidth=2, linestyle='--', alpha=0.6, label='Linear regression')
ax.plot(x_range, y_prob, color='blue', linewidth=2.5, label='Logistic regression')
ax.axhline(0.5, color='red', linestyle=':', alpha=0.5, label='Threshold = 0.5')
ax.set_xlabel('Hours Studied')
ax.set_ylabel('P(Pass)')
ax.set_title('Linear vs Logistic Regression for Classification')
ax.legend()
ax.set_ylim(-0.15, 1.15)
plt.tight_layout()
plt.show()

# Find the decision boundary (where P = 0.5)
boundary = -log_reg.intercept_[0] / log_reg.coef_[0][0]
print(f'Decision boundary: {boundary:.2f} hours')
print(f'Students who study more than {boundary:.1f} hours are predicted to pass')

## Feature Scaling Matters

Logistic regression uses gradient-based optimization internally. When features are on
very different scales, convergence can be slow and coefficient magnitudes become
misleading. **Always scale your features before fitting logistic regression.**

In [ ]:
# Generate data with features on very different scales
X_raw, y_raw = make_classification(
    n_samples=300, n_features=2, n_informative=2, n_redundant=0,
    random_state=42
)
# Scale feature 1 to a very different range
X_skewed = X_raw.copy()
X_skewed[:, 1] = X_skewed[:, 1] * 1000

# Without scaling
model_noscale = LogisticRegression(random_state=42, max_iter=1000)
model_noscale.fit(X_skewed, y_raw)

# With scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_skewed)
model_scaled = LogisticRegression(random_state=42, max_iter=1000)
model_scaled.fit(X_scaled, y_raw)

print('Without scaling:')
print(f'  Coefficients: {model_noscale.coef_[0]}')
print(f'  Accuracy: {model_noscale.score(X_skewed, y_raw):.4f}')
print()
print('With StandardScaler:')
print(f'  Coefficients: {model_scaled.coef_[0]}')
print(f'  Accuracy: {model_scaled.score(X_scaled, y_raw):.4f}')

> After scaling, the coefficients are on comparable scales and reflect the true importance
> of each feature. Without scaling, the coefficient for the large-scale feature is tiny
> (to compensate for its large values), making interpretation difficult.

## Key Takeaways

| Concept | Detail |
|---|---|
| **Sigmoid function** | $\sigma(z) = 1/(1+e^{-z})$ maps any real number to (0, 1) |
| **Log-odds** | Logistic regression is linear in log-odds space |
| **Decision boundary** | The hyperplane where $\mathbf{w}^T\mathbf{x} = 0$ (probability = 0.5) |
| **MLE / Log-loss** | Finds weights that maximize the likelihood of observed labels |
| **Coefficients** | Each $w_j$ is the change in log-odds per unit change in $x_j$ |
| **predict_proba()** | Returns class probabilities, not just hard labels |
| **Threshold** | Default is 0.5; adjusting it trades precision for recall |
| **Feature scaling** | Essential for convergence and interpretable coefficients |

---

**Next up:** Evaluation metrics (confusion matrix, ROC, AUC) and model tuning.